# 11 — Step 9: retrieval for free-text requests

**Purpose.** A user types something like *"a soft, soothing rom-com like How to Lose a Guy in 10 Days, urban, working people"*, and VibeRec must return the right movies **from the catalog** (PROJECT.md §8: always retrieve from the catalog first). **Pass/fail check (roadmap):** retrieval quality on test queries.

**What Step 5 already showed.** Raw document similarity does not find "feel": embedding search scored 0.22 against 0.37 for genre. So a request is first **translated into the 41 vibe dimensions**, then matched against each movie's vibe vector. Keyword search covers literal words.

**Methods compared** (free only; a paid LLM parser is optional and not tested here)

| Method | How a request becomes a ranking |
|---|---|
| **BM25** (baseline) | classic keyword search over plot + reception: movies containing the request's words, rarer words counting more |
| **Doc embedding** | request meaning vs each movie's plot/reception meaning (the Step 5 Test B approach, here as a search baseline) |
| **Vibe parse: text model** | request text → the Step 5 TF-IDF + ridge model → 41 vibe scores → movies with the closest vibe |
| **Vibe parse: embedding** | request meaning vs each vibe's description ("a movie that is dark, bleak, grim…"), giving a vibe profile → closest movies. Handles words never seen in training ("soothing") |
| **Fusion** | each vibe parser combined with BM25 by reciprocal rank fusion (a movie ranked high by either rises) |

**Automatic test (no peeking)**
- **Movies:** the 2,511 held-out movies from Step 5. Their vibe vectors are text-model predictions; the model never saw their tags or Genome scores.
- **Queries:** MovieLens user tags applied by at least 2 different users to at least 10 of these movies (340 tags). Relevant movies are those tagged so.
- **Metrics:** precision@10, recall@50, ndcg@10, averaged over queries. Reported separately for **vibe queries** (tags that are in the vibe taxonomy) and **other queries** (topics, names, eras).

**Human test.** Adil's own requests, rated by him (sheet prepared at the end).

## 1. Load

In [ ]:
import json
import sys
import time
from datetime import datetime, timezone
from pathlib import Path

import joblib
import numpy as np
import pandas as pd

sys.path.insert(0, str(Path("..").resolve()))
from src.viberec import evaluation as ev, genome as gn, retrieval as rt, text_model as tm

PROCESSED_DIR, ARTIFACTS_DIR, MODELS_DIR = Path("..") / "data" / "processed", Path("..") / "artifacts", Path("..") / "models"
ML_DIR = Path("..") / "data" / "raw" / "movielens" / "ml-25m"
SCOREBOARD = ARTIFACTS_DIR / "scoreboard.csv"
VERSION, SEED = "v1", 42


def write_metadata(path, **info):
    meta = {"created_at": datetime.now(timezone.utc).isoformat(), "produced_by": "notebooks/11_retrieval.ipynb", **info}
    path.with_name(path.stem + ".metadata.json").write_text(json.dumps(meta, indent=2, default=str))


wt = pd.read_parquet(PROCESSED_DIR / f"wiki_text_{VERSION}.parquet").set_index("movieId")
genome_ids = pd.read_parquet(PROCESSED_DIR / f"genome_matrix_{VERSION}.parquet", columns=[]).index
model_ids = wt.index[wt["plot_words"] > 0].intersection(genome_ids)
_, pool_ids = tm.split_ids(model_ids, test_share=0.2, seed=42)              # the Step 5 held-out movies
text_model = joblib.load(MODELS_DIR / f"tfidf_ridge_plot_reception_{VERSION}.joblib")
ATTRS = [a for d in gn.VIBE_MAP.values() for a in d if a not in gn.NOT_GENOME_CHECKABLE]
pool_text = [tm.movie_text(wt.at[i, "plot"], wt.at[i, "reception"]) for i in pool_ids]
pool_vibes = text_model.predict(pool_text)                                  # text-model vibe scores, 2,511 x 41
v_mean, v_std = pool_vibes.mean(axis=0), pool_vibes.std(axis=0)
pool_z = rt.standardise_rows(pool_vibes, v_mean, v_std)
print(f"{len(pool_ids):,} pool movies, {pool_vibes.shape[1]} vibes")

In [ ]:
tags = pd.read_csv(ML_DIR / "tags.csv", usecols=["userId", "movieId", "tag"])
tags["tag"] = tags["tag"].astype(str).str.lower().str.strip()
tags = tags[tags["movieId"].isin(pool_ids)]
per = tags.groupby(["tag", "movieId"])["userId"].nunique().reset_index()
per = per[per["userId"] >= 2]                                               # at least two people agree
relevant = per.groupby("tag")["movieId"].apply(set)
relevant = relevant[relevant.map(len) >= 10]
vibe_words = set(gn.map_table()["tag"])
qtype = pd.Series({q: ("vibe" if q in vibe_words else "other") for q in relevant.index})
print(f"{len(relevant)} query tags: {qtype.value_counts().to_dict()}; relevant movies per query: median {relevant.map(len).median():.0f}")
print("vibe queries:", ", ".join(qtype.index[qtype == "vibe"][:40]))

## 2. The methods

The embedding model is the Step 5 `bge-small-en-v1.5`, already on disk. Search queries get bge's query instruction; vibe descriptions and movie texts do not.

In [ ]:
from sentence_transformers import SentenceTransformer
import torch

encoder = SentenceTransformer("BAAI/bge-small-en-v1.5", device="mps" if torch.backends.mps.is_available() else "cpu")
QUERY_PREFIX = "Represent this sentence for searching relevant passages: "
attr_tags = gn.map_table().groupby("attribute")["tag"].apply(list)
attr_texts = ["a movie that is " + ", ".join(attr_tags[a]) for a in ATTRS]
attr_vecs = encoder.encode(attr_texts, normalize_embeddings=True)

emb_ids = np.load(ARTIFACTS_DIR / f"embeddings_bge-small_movieIds_{VERSION}.npy")
row = pd.Series(np.arange(len(emb_ids)), index=emb_ids).loc[pool_ids].to_numpy()
doc_vec = np.load(ARTIFACTS_DIR / f"embeddings_bge-small_plot_{VERSION}.npy")[row] + np.load(ARTIFACTS_DIR / f"embeddings_bge-small_reception_{VERSION}.npy")[row]
doc_vec /= np.linalg.norm(doc_vec, axis=1, keepdims=True)
bm25 = rt.BM25(pool_text)


def method_scores(query):
    """One score per pool movie for each method."""
    s = {"bm25": bm25.scores(query)}
    s["doc_embedding"] = doc_vec @ encoder.encode([QUERY_PREFIX + query], normalize_embeddings=True)[0]
    q_tm = rt.standardise_rows(text_model.predict([query])[0], v_mean, v_std)
    s["vibe_text_model"] = rt.cosine_rank_scores(pool_z, q_tm)
    s["vibe_embedding"] = rt.cosine_rank_scores(pool_z, rt.embedding_vibe_profile(encoder, query, attr_texts, attr_vecs))
    s["fusion_text_model+bm25"] = rt.rrf([s["vibe_text_model"], s["bm25"]])
    s["fusion_embedding+bm25"] = rt.rrf([s["vibe_embedding"], s["bm25"]])
    return s

## 3. Automatic test on the 340 tag queries

In [ ]:
RET_PATH = ARTIFACTS_DIR / f"retrieval_eval_{VERSION}.parquet"
if RET_PATH.exists():
    print(f"{RET_PATH} already exists, loading.")
    ret = pd.read_parquet(RET_PATH)
else:
    start, rows = time.time(), []
    for q, rel in relevant.items():
        for method, s in method_scores(q).items():
            order = pool_ids[np.argsort(-s)]
            rows.append({"query": q, "type": qtype[q], "method": method, "relevant": len(rel),
                         "precision@10": rt.precision_at_k(list(order), rel, 10),
                         "recall@50": ev.recall_at_k(list(order), rel, 50), "ndcg@10": ev.ndcg_at_k(list(order), rel, 10)})
    ret = pd.DataFrame(rows)
    ret.to_parquet(RET_PATH, index=False)
    write_metadata(RET_PATH, queries=len(relevant), pool="Step 5 held-out movies (2,511)",
                   relevance="tagged by >= 2 users; queries with >= 10 relevant movies", runtime_seconds=round(time.time() - start, 1))

table = ret.groupby(["type", "method"])[["precision@10", "recall@50", "ndcg@10"]].mean().round(4)
print(table.to_string())
# chance level: a random ranking's expected precision@10 = share of the pool that is relevant
print("\nchance precision@10 (random order):", ret.groupby("type").apply(lambda d: (d["relevant"] / len(pool_ids)).mean()).round(4).to_dict())
idx = ret.set_index(["method", "query"])
for t in ("vibe", "other"):
    qs = qtype.index[qtype == t]
    for m in ("vibe_text_model", "vibe_embedding", "fusion_text_model+bm25", "fusion_embedding+bm25", "doc_embedding"):
        d, lo, hi, rel_ = ev.paired_gain(idx.loc[m].loc[qs], idx.loc["bm25"].loc[qs], "precision@10", seed=SEED)
        print(f"{t:5s} queries: {m:24s} vs BM25 precision@10 {rel_:+.0%} [{lo:+.4f}, {hi:+.4f}]")
for m, d in ret.groupby("method"):
    res = {f"{t}_{k}": d.loc[d["type"] == t, k].mean() for t in ("vibe", "other") for k in ("precision@10", "recall@50", "ndcg@10")}
    ev.append_scoreboard(SCOREBOARD, "9 retrieval", f"search: {m}", res,
                         settings={"pool": "Step 5 held-out movies", "pool_size": len(pool_ids), "queries": len(relevant), "seed": SEED},
                         note="MovieLens tag queries (>= 2 users, >= 10 movies)")

**Results (2,511 held-out movies; 82 vibe queries, 258 other queries)**

Precision@10 = share of the top 10 that users actually tagged with the query. A random order gives about 0.01.

| Method | Vibe queries | Other queries (topics, names, eras) |
|---|---|---|
| BM25 (keywords) | 0.127 | **0.183** |
| Doc embedding | 0.111 | 0.173 (≈ BM25) |
| Vibe parse: text model | 0.096 (−24%) | 0.059 |
| Vibe parse: embedding | 0.112 (−12%, n.s.) | 0.043 |
| Fusion: text-model parse + BM25 | 0.134 (+6%, n.s.) | 0.132 (−28%) |
| **Fusion: embedding parse + BM25** | **0.161 (+27%** [+0.011, +0.057]) | 0.135 (−26%) |

- **For vibe requests, the winner is the embedding vibe parser fused with keyword search:** +27% over BM25 (significant). Neither part alone beats BM25; together they do. Keywords catch movies whose text literally says "bleak" or "heartwarming"; the vibe parse catches movies that feel that way without saying it.
- **For topic, name and era requests ("courtroom", "brad pitt", "1950s"), keyword search wins,** and adding the vibe parse hurts (−26%). Vibe is the wrong tool for them.
- **The text-model parser is weaker than the embedding parser** for short requests (trained on long plots), so the embedding parser is the one to keep.
- Every method is 10–18× above chance.

**Consequence for the request pipeline.** Requests must be **routed**: vibe parts to the fused vibe search, literal parts (topics, names, a reference movie) to keyword or title search. A real request mixes both ("soft, soothing rom-com … urban, working people"), which is why PROJECT.md §8 splits a request into parts first.

**Limitations.**
- The tag queries are single concepts. Full mixed requests need the human test.
- Relevance comes from MovieLens taggers, so an untagged but fitting movie counts as a miss; absolute precision is a lower bound.
- The "vibe" vs "other" label of a query comes from the taxonomy's word list.

## 4. Splitting a full request into parts

`src/viberec/request.py` splits a request into:
- **reference movie:** a catalog title inside the request; one-word titles only after "like" / "similar to";
- **genres:** a small synonym list, e.g. "rom-com" → Romance + Comedy (vocabulary, not a tuned number);
- **vibe phrases vs literal phrases:** the rest is cut at commas, "and", "with", "about", filler words dropped. A **router** decides per phrase whether it describes a feeling or something literal.

**The router** is logistic regression on the phrase's embedding, trained on the 340 tag queries labelled vibe / other. Because it works from meaning, it can route phrases it never saw ("soothing"). Its accuracy is measured with 5-fold cross-validation (each tag predicted by a model that did not see it), against always guessing the majority class ("other", 76%).

**Router training data (revised after the first run).** Trained on the tag queries alone, the router caught only 34% of vibe phrases (it sent "heartbreaking" to keyword search). Every word in the vibe taxonomy is a vibe word by definition, so those words are added as extra vibe examples, and the two classes are weighted equally.
- Cross-validated (folds over the tag queries; taxonomy words only ever in training): **vibe phrases caught 70%** (was 34%), **literal phrases kept 76%** (was 95%).
- VibeRec requests are expected to be mostly about feel, so catching vibe words is favoured. The human test measures whether that trade-off pays off.

In [ ]:
from sklearn.linear_model import LogisticRegressionCV
from sklearn.model_selection import StratifiedKFold
from src.viberec import request as rq

tag_texts = list(relevant.index)
tag_y = (qtype.loc[tag_texts] == "vibe").astype(int).to_numpy()
extra_vibe = sorted(set(gn.map_table()["tag"]) - set(tag_texts))           # taxonomy words: vibe by definition
x_tags = encoder.encode(tag_texts, normalize_embeddings=True)
x_extra = encoder.encode(extra_vibe, normalize_embeddings=True)
for balanced, extra in ((False, False), (True, True)):
    pred = np.zeros(len(tag_texts), int)
    for trn, tst in StratifiedKFold(5, shuffle=True, random_state=SEED).split(x_tags, tag_y):
        X = np.vstack([x_tags[trn], x_extra]) if extra else x_tags[trn]
        Y = np.concatenate([tag_y[trn], np.ones(len(extra_vibe), int)]) if extra else tag_y[trn]
        pred[tst] = LogisticRegressionCV(Cs=10, cv=5, max_iter=5000, class_weight="balanced" if balanced else None).fit(X, Y).predict(x_tags[tst])
    label = "tags + taxonomy words, balanced" if extra else "tags only"
    print(f"router ({label}): accuracy {(pred == tag_y).mean():.1%}; vibe phrases caught {pred[tag_y == 1].mean():.0%}; "
          f"literal phrases kept literal {1 - pred[tag_y == 0].mean():.0%}")
router = rq.Router(encoder).fit(tag_texts + extra_vibe, np.concatenate([tag_y, np.ones(len(extra_vibe), int)]), balanced=True)
for w in ["soothing", "cozy", "soft", "heartbreaking", "slow burn", "feel good", "working people", "urban life", "courtroom", "brad pitt", "1950s"]:
    print(f"   {w:15s} P(vibe) = {router.p_vibe([w])[0]:.2f}")

## 5. The search engine over the whole catalog

**Catalog:** approved (tier 3) movies with vibe scores (`vibe_scores_v2`). Text for keyword search: Wikipedia plot + reception, or the TMDB overview + keywords where there is no Wikipedia page.

**Each part of the request becomes a ranking; the rankings are fused by reciprocal rank fusion:**
- **vibe phrases** → embedding vibe parser fused with BM25 (the tested winner for vibe requests);
- **literal phrases** → BM25 (the tested winner for topics and names);
- **reference movie** → movies closest to it in vibe (41 vibe scores) and in taste (ALS factors). This part is not yet tested on its own; the human test covers it;
- **genres** → a filter: results must have all named genres.

The reference movie itself is never returned. The output keeps which parts each result matched, for the Torch button later.

In [ ]:
tiers_df = pd.read_parquet(PROCESSED_DIR / "movie_tiers_v2.parquet").set_index("movieId")
vibe_all = pd.read_parquet(PROCESSED_DIR / "vibe_scores_v2.parquet").set_index("movieId")
cat_ids = vibe_all.index[vibe_all.index.isin(tiers_df.index[tiers_df["tier"] == 3])].to_numpy()
cat_v = vibe_all.loc[cat_ids, ATTRS].astype(float).to_numpy()
cat_z = (cat_v - cat_v.mean(axis=0)) / cat_v.std(axis=0)
tmdb_dir = Path("..") / "data" / "raw" / "tmdb" / f"movie_{VERSION}"
tmdb_of = tiers_df["tmdbId"].dropna().astype(int)


def catalog_text(m):
    if m in wt.index:
        return tm.movie_text(wt.at[m, "plot"], wt.at[m, "reception"])
    f = tmdb_dir / f"{tmdb_of.get(m, -1)}.json"
    d = json.loads(f.read_text()) if f.exists() else {}
    return (d.get("overview") or "") + " " + " ".join(d.get("keywords") or [])


cat_bm25 = rt.BM25([catalog_text(m) for m in cat_ids])
cat_genres = tiers_df.loc[cat_ids, "genres"].fillna("").str.split("|")
als_items = np.load(MODELS_DIR / f"als_item_factors_{VERSION}.npy")
als_index = pd.Series(np.arange(len(als_items)), index=pd.read_parquet(MODELS_DIR / f"als_movie_index_{VERSION}.parquet")["movieId"])
cat_als = np.vstack([als_items[als_index[m]] if m in als_index.index else np.zeros(als_items.shape[1]) for m in cat_ids])
title_index = rq.build_title_index(pd.read_csv(ML_DIR / "movies.csv"), set(cat_ids), tiers_df["rating_count"].to_dict())
cat_pos = pd.Series(np.arange(len(cat_ids)), index=cat_ids)


def search(request, k=10):
    parsed = rq.parse_request(request, title_index, router)
    rankings, parts = [], []
    if parsed["vibe_phrases"]:
        vq = " ".join(parsed["vibe_phrases"])
        prof = rt.embedding_vibe_profile(encoder, vq, attr_texts, attr_vecs)
        rankings.append(rt.rrf([rt.cosine_rank_scores(cat_z, prof), cat_bm25.scores(vq)]))
        parts.append("vibe")
    if parsed["literal_phrases"]:
        rankings.append(cat_bm25.scores(" ".join(parsed["literal_phrases"])))
        parts.append("literal")
    ref = parsed["reference"]
    if ref is not None and ref in cat_pos.index:
        r = cat_pos[ref]
        a = cat_als[r]
        als_sim = cat_als @ a / (np.linalg.norm(cat_als, axis=1) * (np.linalg.norm(a) or 1) + 1e-9)
        rankings.append(rt.rrf([rt.cosine_rank_scores(cat_z, cat_z[r]), als_sim]))
        parts.append("reference")
    if not rankings:
        return parsed, []
    score = rt.rrf(rankings) if len(rankings) > 1 else rankings[0]
    if parsed["genres"]:
        ok = cat_genres.map(lambda g: all(x in g for x in parsed["genres"])).to_numpy()
        score = np.where(ok, score, -np.inf)
    if ref is not None and ref in cat_pos.index:
        score[cat_pos[ref]] = -np.inf
    top = np.argsort(-score)[:k]
    return parsed, [(int(cat_ids[i]), tiers_df.at[cat_ids[i], "title"]) for i in top if np.isfinite(score[i])]


EXAMPLES = ["I wanna watch a soft, soothing rom-com like How to Lose a Guy in 10 Days, based on urban life, working people",
            "something dark and bleak about revenge",
            "a cozy feel-good animated movie for a rainy day",
            "mind-bending sci-fi like Inception",
            "a heartbreaking coming of age drama set in a small town"]
for req in EXAMPLES:
    parsed, res = search(req)
    ref_title = tiers_df.at[parsed["reference"], "title"] if parsed["reference"] else None
    print(f"\nREQUEST: {req}\n  parsed: reference={ref_title}, genres={parsed['genres']}, vibe={parsed['vibe_phrases']}, literal={parsed['literal_phrases']}")
    for rank, (m, t) in enumerate(res, 1):
        print(f"  {rank:2d}. {t}")

## 6. Human test: Adil's 30 requests

Adil wrote 30 requests (`data/labels/search_requests_v1.xlsx`), each naming a reference movie, plus a movie he would expect.

**Automatic check:** does the expected movie appear in the top 10 or top 50? Expected movies not in the catalog (released after MovieLens ends in 2019, or not approved) are flagged, not counted as misses.

**Human check:** the top 10 for every request go into `data/labels/search_results_v1.xlsx`, a new file; the request sheet is not changed. Adil marks each result **fits / sort of / doesn't fit**.

**Parser fixes found by this test (before rating).** The first run (`search_results_v1.xlsx`, not to be rated) showed four parsing bugs:
- "a superhero movie like Iron Man" matched the film *Superhero Movie*; "a fun murder mystery like Knives Out" matched the film *Murder Mystery*. Fix: a title right after "like" / "similar to" now wins.
- A curly apostrophe broke "We’re the Millers". Fix: curly quotes are turned into straight ones.
- "less action" and "Animation is fine" became required genres. Fix: genre words after less / no / not / without, or followed by "is fine" / "is ok", are not required.

The rating sheet is `search_results_v2.xlsx`.

In [ ]:
import re
from openpyxl import load_workbook

reqs = pd.read_excel(Path("..") / "data" / "labels" / "search_requests_v1.xlsx", sheet_name="Requests").dropna(subset=["Your request (as you would type it)"])
reqs.columns = ["n", "request", "expected"]
ml_titles = pd.read_csv(ML_DIR / "movies.csv")
any_index = rq.build_title_index(ml_titles, set(ml_titles["movieId"]), tiers_df["rating_count"].to_dict())   # all MovieLens titles


def find_title(title):
    """Expected-movie lookup: a year in brackets, e.g. 'Robin Hood (2010)', is dropped before matching."""
    key = rq.norm(re.sub(r"\s*\(\d{4}\)\s*$", "", str(title)))
    return any_index.get(key)


rows = []
for r in reqs.itertuples():
    parsed, res = search(r.request, k=50)
    ids = [m for m, _ in res]
    exp_id = find_title(r.expected) if isinstance(r.expected, str) else None
    status = ("not in MovieLens" if exp_id is None else "not in catalog (not approved / no vibe scores)"
              if exp_id not in cat_pos.index else "in catalog")
    rank = ids.index(exp_id) + 1 if exp_id in ids else None
    rows.append({"n": r.n, "request": r.request, "reference": tiers_df["title"].get(parsed["reference"]),
                 "genres": ", ".join(parsed["genres"]), "vibe": "; ".join(parsed["vibe_phrases"]),
                 "literal": "; ".join(parsed["literal_phrases"]), "expected": r.expected, "expected_status": status,
                 "expected_rank_top50": rank, "top10": [t for _, t in res[:10]]})
human = pd.DataFrame(rows)
found = human[human["expected_status"] == "in catalog"]
print(f"reference movie recognised in {human['reference'].notna().sum()} of {len(human)} requests")
print(f"expected movie in the catalog for {len(found)} of {len(human)}; of those: top 10 {(found['expected_rank_top50'] <= 10).sum()}, "
      f"top 50 {found['expected_rank_top50'].notna().sum()}")
print(human[["n", "reference", "genres", "expected", "expected_status", "expected_rank_top50"]].to_string(index=False))

RESULTS_XLSX = Path("..") / "data" / "labels" / "search_results_v2.xlsx"   # v2: after the parser fixes
if RESULTS_XLSX.exists():
    print(f"\n{RESULTS_XLSX} already exists (Adil may have rated it); not overwritten.")
else:
    long = pd.DataFrame([{"request #": r.n, "request": r.request if i == 0 else "", "rank": i + 1, "result": t, "rating": ""}
                         for r in human.itertuples() for i, t in enumerate(r.top10)])
    long.to_excel(RESULTS_XLSX, sheet_name="Rate the results", index=False)
    wb = load_workbook(RESULTS_XLSX)
    ws = wb["Rate the results"]
    from openpyxl.worksheet.datavalidation import DataValidation
    from openpyxl.styles import Font, Alignment
    dv = DataValidation(type="list", formula1='"fits,sort of,doesn\'t fit"', allow_blank=True)
    ws.add_data_validation(dv)
    dv.add(f"E2:E{ws.max_row}")
    for col, w in zip("ABCDE", (10, 70, 6, 55, 14)):
        ws.column_dimensions[col].width = w
    for c in ws[1]:
        c.font = Font(bold=True)
    for row in ws.iter_rows(min_row=2):
        row[1].alignment = Alignment(wrap_text=True, vertical="top")
    ws.freeze_panes = "A2"
    wb.save(RESULTS_XLSX)
    print(f"\nrating sheet written: {RESULTS_XLSX} ({len(long)} results to rate)")
human.drop(columns="top10").to_csv(ARTIFACTS_DIR / "human_search_check_v2.csv", index=False)

## 7. Popularity-aware search (after Adil's feedback and the Step 10 ranker)

**Problem.** Adil had not seen most results (44% of the top 10 had under 1,000 ratings). The search matched vibe and words but had no sense of how much people watch a film. The Step 10 ranker found that **present-day popularity** is the strongest signal of what users want.

**Change.** Present-day popularity becomes one more ranking in the fusion, next to the vibe, keyword and reference rankings. It is combined by the same reciprocal rank fusion, so it gets an equal vote and no hand-set weight.
- **"Present-day":** ratings in the last ~18.5 months of the data, the same window length as the ranker (the gap between the 2016 and 2018 cutoffs), ending at the last rating in MovieLens (November 2019).

**Kept only if both hold:**
1. **Familiarity improves** on Adil's 30 requests: the share of top-10 results with under 1,000 ratings, and their median rating count.
2. **Relevance does not clearly drop** on the 340 tag queries: precision@10 against the vibe fusion, paired by query.

Note: tags are applied more often to popular films, so the tag test may favour popularity a little.

In [ ]:
REC_PATH = PROCESSED_DIR / f"recent_popularity_latest_{VERSION}.parquet"
if REC_PATH.exists():
    print(f"{REC_PATH} already exists, loading.")
    recent_latest = pd.read_parquet(REC_PATH).set_index("movieId")["recent_ratings"]
else:
    early_all = pd.read_parquet(ARTIFACTS_DIR / f"q43_early_counts_{VERSION}.parquet")
    window = int(early_all.loc[early_all["quantile"] == 0.9, "cutoff_ts"].iloc[0]) - int(early_all.loc[early_all["quantile"] == 0.8, "cutoff_ts"].iloc[0])
    allr = pd.read_csv(ML_DIR / "ratings.csv", usecols=["movieId", "timestamp"], dtype={"movieId": "int32", "timestamp": "int64"})
    end = int(allr["timestamp"].max())
    recent_latest = allr[allr["timestamp"] >= end - window].groupby("movieId").size().rename("recent_ratings")
    recent_latest.reset_index().to_parquet(REC_PATH, index=False)
    write_metadata(REC_PATH, window_days=round(window / 86400), window_end=str(pd.to_datetime(end, unit="s")),
                   note="ratings per movie in the last ~18.5 months of MovieLens: present-day popularity")
    del allr
print(f"{(recent_latest > 0).sum():,} movies rated in the last window; most watched: "
      + ", ".join(tiers_df["title"].reindex(recent_latest.nlargest(5).index).fillna("?")))

cat_recent = np.log1p(recent_latest.reindex(cat_ids).fillna(0).to_numpy())


def search_v2(request, k=10, popularity=True):
    """search() plus present-day popularity as one more ranking in the fusion."""
    parsed = rq.parse_request(request, title_index, router)
    rankings = []
    if parsed["vibe_phrases"]:
        vq = " ".join(parsed["vibe_phrases"])
        prof = rt.embedding_vibe_profile(encoder, vq, attr_texts, attr_vecs)
        rankings.append(rt.rrf([rt.cosine_rank_scores(cat_z, prof), cat_bm25.scores(vq)]))
    if parsed["literal_phrases"]:
        rankings.append(cat_bm25.scores(" ".join(parsed["literal_phrases"])))
    ref = parsed["reference"]
    if ref is not None and ref in cat_pos.index:
        r = cat_pos[ref]
        a = cat_als[r]
        als_sim = cat_als @ a / (np.linalg.norm(cat_als, axis=1) * (np.linalg.norm(a) or 1) + 1e-9)
        rankings.append(rt.rrf([rt.cosine_rank_scores(cat_z, cat_z[r]), als_sim]))
    if not rankings:
        return parsed, []
    if popularity:
        rankings.append(cat_recent)
    score = rt.rrf(rankings) if len(rankings) > 1 else rankings[0]
    if parsed["genres"]:
        ok = cat_genres.map(lambda g: all(x in g for x in parsed["genres"])).to_numpy()
        score = np.where(ok, score, -np.inf)
    if ref is not None and ref in cat_pos.index:
        score[cat_pos[ref]] = -np.inf
    top = np.argsort(-score)[:k]
    return parsed, [(int(cat_ids[i]), tiers_df.at[cat_ids[i], "title"]) for i in top if np.isfinite(score[i])]


# 1. familiarity on Adil's 30 requests
counts = tiers_df["rating_count"]
fam = {}
for label, pop in (("before (no popularity)", False), ("with present-day popularity", True)):
    ids = [m for req in reqs["request"] for m, _ in search_v2(req, k=10, popularity=pop)[1]]
    c = counts.reindex(ids)
    fam[label] = {"median ratings": int(c.median()), "share under 1,000 ratings": f"{(c < 1000).mean():.0%}",
                  "share with recent ratings": f"{(recent_latest.reindex(ids).fillna(0) > 0).mean():.0%}"}
print(pd.DataFrame(fam).T.to_string())

# 2. relevance on the 340 tag queries (pool = Step 5 held-out movies)
pool_recent = np.log1p(recent_latest.reindex(pool_ids).fillna(0).to_numpy())
rows_p = []
for q, rel in relevant.items():
    s = method_scores(q)
    for name, sc in {"fusion_embedding+bm25": s["fusion_embedding+bm25"],
                     "fusion + popularity": rt.rrf([s["vibe_embedding"], s["bm25"], pool_recent]),
                     "bm25 + popularity": rt.rrf([s["bm25"], pool_recent])}.items():
        order = list(pool_ids[np.argsort(-sc)])
        rows_p.append({"query": q, "type": qtype[q], "method": name, "precision@10": rt.precision_at_k(order, rel, 10)})
pp = pd.DataFrame(rows_p)
print()
print(pp.groupby(["type", "method"])["precision@10"].mean().round(4).to_string())
pidx = pp.set_index(["method", "query"])
for t, base in (("vibe", "fusion_embedding+bm25"), ("other", "fusion_embedding+bm25")):
    qs = qtype.index[qtype == t]
    for m in ("fusion + popularity", "bm25 + popularity"):
        d, lo, hi, rel_ = ev.paired_gain(pidx.loc[m].loc[qs], pidx.loc[base].loc[qs], "precision@10", seed=SEED)
        print(f"{t:5s} queries: {m:20s} vs {base}: {rel_:+.0%} [{lo:+.4f}, {hi:+.4f}]")

In [ ]:
# the same requests as before, now with present-day popularity in the fusion
for req in [EXAMPLES[0], EXAMPLES[1], EXAMPLES[2]] + list(reqs["request"].iloc[[0, 4, 11, 12, 22]]):
    parsed, res = search_v2(req, k=10)
    print(f"\nREQUEST: {req}")
    print("  " + " · ".join(t for _, t in res))

### Popularity as an equal vote is too strong, so it only reorders relevant films

The examples above show the cost of an equal vote: famous films that do not fit the request (*Inception* and *The Matrix* for "dark and bleak about revenge") enter the top 10 on popularity alone.

**Change:** a two-stage search.
1. **Relevance first:** the top `N_RELEVANT` films by the relevance fusion alone (vibe, keywords, reference). An off-topic blockbuster cannot get in.
2. **Then popularity reorders only those:** fusion of the relevance ranking and present-day popularity.

`N_RELEVANT` = 50, the same depth as recall@50 in the testing zone. 25 and 100 are also reported to show how sensitive the result is to it. Checked on familiarity (Adil's requests), tag precision, and the examples.

In [ ]:
def search_v3(request, k=10, n_relevant=50):
    """Two-stage: top n_relevant by relevance only, then reordered by relevance + present-day popularity."""
    parsed, rel = search_v2(request, k=len(cat_ids), popularity=False)
    if not rel:
        return parsed, []
    top_ids = [m for m, _ in rel[:n_relevant]]
    pos = cat_pos.loc[top_ids].to_numpy()
    rel_score = -np.arange(len(top_ids), dtype=float)                       # relevance order
    order = np.argsort(-rt.rrf([rel_score, cat_recent[pos]]))[:k]
    return parsed, [(top_ids[i], tiers_df.at[top_ids[i], "title"]) for i in order]


def two_stage_scores(rel_scores, pop, n):
    """Tag-benchmark version: keep the top n by relevance, reorder them with popularity, push the rest below."""
    out = np.full(len(rel_scores), -np.inf)
    top = np.argsort(-rel_scores)[:n]
    rel_rank = np.empty(len(top)); rel_rank[:] = -np.arange(len(top))
    out[top] = rt.rrf([rel_rank, pop[top]]) + 10                            # +10 keeps them above everything else
    rest = np.setdiff1d(np.arange(len(rel_scores)), top)
    out[rest] = rel_scores[rest] - rel_scores.max() - 1
    return out


fam3, prec3 = {}, []
for n in (25, 50, 100):
    ids = [m for req in reqs["request"] for m, _ in search_v3(req, k=10, n_relevant=n)[1]]
    c = counts.reindex(ids)
    fam3[f"two-stage, top {n}"] = {"median ratings": int(c.median()), "share under 1,000 ratings": f"{(c < 1000).mean():.0%}"}
    for q, rel in relevant.items():
        s = method_scores(q)
        order = list(pool_ids[np.argsort(-two_stage_scores(s["fusion_embedding+bm25"], pool_recent, n))])
        prec3.append({"query": q, "type": qtype[q], "method": f"two-stage, top {n}", "precision@10": rt.precision_at_k(order, rel, 10)})
fam3 = {**fam, **fam3}
print(pd.DataFrame(fam3).T.to_string())
pp3 = pd.concat([pp, pd.DataFrame(prec3)])
print()
print(pp3.groupby(["type", "method"])["precision@10"].mean().unstack(0).round(4).to_string())
for req in [EXAMPLES[1], EXAMPLES[2]] + list(reqs["request"].iloc[[11, 22]]):
    print(f"\nREQUEST: {req[:90]}\n  " + " · ".join(t for _, t in search_v3(req)[1]))

**Results: popularity-aware search**

| Version | Results under 1,000 ratings (Adil's requests) | Median ratings | Tag precision@10, vibe | Tag precision@10, other |
|---|---|---|---|---|
| no popularity | 53% | 726 | 0.161 | 0.135 |
| popularity as an equal vote | 18% | 22,943 | 0.213 | 0.206 |
| two-stage, top 25 | 30% | 4,385 | 0.223 | 0.191 |
| **two-stage, top 50** (chosen) | **19%** | 8,270 | **0.240** | **0.214** |
| two-stage, top 100 | 11% | 10,346 | 0.248 | 0.232 |

- **Popularity as an equal vote** made results familiar but let off-topic blockbusters in (*Inception*, *The Matrix* for "dark and bleak about revenge").
- **Two-stage** keeps results on topic and familiar: the *Forrest Gump* request now returns *The Shawshank Redemption*, *The Green Mile*, *Slumdog Millionaire*, *Hachiko*, *The Intouchables*, *Good Will Hunting*.
- **Top 50 is kept as pre-set.** Top 100 scored better on both measures, but it is the edge of the grid and was picked on these same results. It is re-checked on Adil's new requests rather than adopted now.

**Still weak**
- Requests whose relevant films are mostly obscure (dark revenge).
- Requests that name a reference film without "like" ("Titanic. Intense chemistry…", "Hangover, the more similar the better"). The rule-based parser misses the reference, which is the case for testing a Bedrock (LLM) parser.